<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# The training loop

**[The training loop](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/)** · Deep Learning and Transformers Explained · Module 3, lesson 4 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** train the ticket network with the five steps of the training loop, in the right order, log the loss, and recognise what goes wrong when a step is missing.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account, no graphics card. Training takes a few seconds on a CPU. |
| **You should know** | The loss, gradient descent and `loss.backward()`, from the first three lessons of this module, for example [Backpropagation and autograd](https://learning.nextia-ai.com/courses/deep-learning/m03/backpropagation-and-autograd/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M03-L04-the-training-loop/the-training-loop-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: the tickets as numbers

The next cell turns the tickets into word counts, as in [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/): a vocabulary from the training tickets, then one row of counts per ticket. It also makes `loss_fn`, the loss of this module. Run it.

> **Check.** You should see `x_train shape (4696, 2864)`, `x_valid shape (600, 2864)` and `vocabulary: 2864`.

In [ ]:
import torch
from torch import nn

# The tickets as numbers: a vocabulary from the training tickets, then word counts.
vocab = Vocab([r["text"] for r in train_rows])
x_train, y_train = bag_of_words(train_rows, vocab), labels(train_rows)
x_valid, y_valid = bag_of_words(valid_rows, vocab), labels(valid_rows)
loss_fn = nn.CrossEntropyLoss()
show_shape("x_train", x_train)
show_shape("x_valid", x_valid)
print("vocabulary:", len(vocab), "| teams:", TEAMS)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The five steps

One step of training has five parts, always in this order:

1. `optimizer.zero_grad()`: set every gradient to zero, because `backward()` adds to them.
2. `scores = model(x_train)`: the forward pass.
3. `loss = loss_fn(scores, y_train)`: how wrong the network is.
4. `loss.backward()`: the backward pass, a gradient for every parameter.
5. `optimizer.step()`: move every parameter against its gradient.

An **optimizer** is the object that does steps 1 and 5 for a list of parameters. `torch.optim.SGD(model.parameters(), lr=0.5)` would do exactly the rule of the gradient descent lesson. The course uses `torch.optim.Adam` with a learning rate of 0.001: a version of gradient descent that adapts the step size of each parameter (Module 4 explains how).

This first loop is **full-batch**: every step uses all 4,696 training tickets.

> **Predict.** The loss starts at 1.624. Where will it be after 100 steps: above 1, between 0.5 and 1, or below 0.5? Then run the cell. It takes a few seconds.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for step in range(1, 101):
    optimizer.zero_grad()                  # 1. forget the last step's gradients
    scores = model(x_train)                # 2. forward pass: all 4,696 training tickets
    loss = loss_fn(scores, y_train)        # 3. how wrong?
    loss.backward()                        # 4. backward pass: a gradient for every parameter
    optimizer.step()                       # 5. update every parameter
    if step % 10 == 0:
        print(f"step {step:3}   loss {loss.item():.4f}")

> **Check.** You should see ten lines. The loss is `1.4113` at step 10, `0.8550` at step 30, `0.4454` at step 50, `0.2573` at step 70 and `0.1460` at step 100.

The loss falls at every logged step, fast in the middle and slower at the end. Now measure the trained network on the validation tickets, which it never trained on.

> **Predict.** The untrained network had 18.2% validation accuracy. What will this one have? Then run the cell.

In [ ]:
valid_loss, valid_acc = evaluate(model, x_valid, y_valid)
print(f"validation loss {valid_loss:.4f}   accuracy {valid_acc:.4f}")

> **Check.** You should see `validation loss 0.1985   accuracy 0.9783`.

100 steps took the network from chance to 97.8% on tickets that it never saw.

## 2. Change one thing: leave one step out

The next cell trains for 20 steps and leaves out one of the five steps each time. It prints the loss of the 20th step.

> **Predict.** For each missing step, will the loss at step 20 be the same as with all five steps (`1.1342`), higher, or stuck at the start (`1.6236`)? Then run the cell.

In [ ]:
def train_full_batch(steps, skip=None):
    """The loop above, with one step left out (or none)."""
    set_seed(0)
    model = BagOfWordsNet(len(vocab))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for step in range(steps):
        if skip != "zero_grad":
            optimizer.zero_grad()
        loss = loss_fn(model(x_train), y_train)
        if skip != "backward":
            loss.backward()
        if skip != "step":
            optimizer.step()
    return loss.item()

for skip in [None, "zero_grad", "backward", "step"]:
    print(f"left out: {str(skip):10}   loss at step 20: {train_full_batch(20, skip):.4f}")

> **Check.** You should see `None` 1.1342, `zero_grad` 1.1745, `backward` 1.6236 and `step` 1.6236.

Without `backward()`, there are no gradients, so `step()` has nothing to use. Without `step()`, the gradients are computed and then ignored. Both leave the network as it was, with no error message. Without `zero_grad()`, the network still learns, but worse. The next section shows why.

## 3. Failure case: forgetting `zero_grad()`

`backward()` adds each new gradient to the old ones. Without `zero_grad()`, step 2 uses the gradients of steps 1 and 2 together, step 3 the gradients of steps 1, 2 and 3, and so on.

> **Predict.** The size of the first layer's gradient is 0.1957 at step 1. What happens to it over the next steps without `zero_grad()`? Then run the cell. `.norm()` measures the size of a whole tensor of numbers.

In [ ]:
for skip in [None, "zero_grad"]:
    set_seed(0)
    model = BagOfWordsNet(len(vocab))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    sizes = []
    for step in range(3):
        if skip != "zero_grad":
            optimizer.zero_grad()
        loss_fn(model(x_train), y_train).backward()
        sizes.append(round(model.layers[0].weight.grad.norm().item(), 4))
        optimizer.step()
    print(f"left out: {str(skip):10}   size of the first layer's gradient: {sizes}")

> **Check.** You should see `[0.1957, 0.2066, 0.232]` with all five steps and `[0.1957, 0.3886, 0.5958]` without `zero_grad()`.

Without `zero_grad()`, the gradient piles up: about 0.2 more at every step. The steps are pushed by old gradients that no longer fit the weights. In the course's real setting (small batches of 64 tickets and many steps per pass), Priya measured the same mistake over three passes through the data: the gradient size grew 10.7 → 15.7 → 25.6, and the validation accuracy was only 91.5%, 89.5% and 90.0%, against 97.2% after one pass of the correct loop. **Fix:** call `optimizer.zero_grad()` at the start of every step.

## 4. Check yourself

> **Your turn.** Write the five-step loop with plain gradient descent, `torch.optim.SGD(model.parameters(), lr=0.5)`, for 10 full-batch steps from `set_seed(0)` and a new `BagOfWordsNet`. Append each step's `loss.item()` to the list `losses`. Then run the check cell, which looks at the last value.

In [ ]:
losses = []
# your loop here: set_seed(0), a new model, an SGD optimizer with lr=0.5, 10 steps

In [ ]:
expect('the loss of step 10', losses[-1] if losses else None, 1.4191, tolerance=0.001)

## Next

You can now train a network with the five steps, log its loss, and spot a missing step from its symptom. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/) has the loop diagram, the loss curve and the knowledge checks. The next lesson trains in small batches over several passes, and measures on the validation tickets the right way.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Batches, epochs and validation](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.